# 04 · Error Analysis and Gretel Label Audit
**RedactNER: Local PII Redaction for Word Documents**

This notebook looks behind the headline numbers of notebook 03.
1. **Gretel Label Audit.** Gretel Finance was annotated by an automatic pass rather than by people. Fifty of
   its test documents were reviewed span by span to estimate how much of the measured leak rate is label noise.
2. **What Still Leaks.** The mentions the selected model leaves visible, by category and test set, with
   examples from the real court judgments of TAB.
3. **Disagreements on TAB.** The most frequent text the model redacts that TAB's annotators left visible.
4. **The Operating Point on Real Documents.** How the decision threshold trades leaks for precision on TAB.
5. **Prototype Versus Final Model.** Which categories changed on TAB between the two training sets.

It reads the predictions of the exported model that notebook 03 saved in `outputs/predictions/selected/`.

In [1]:
N_EXAMPLES = 15
SEED = 7
TAU_SENSITIVITY = [0.97, 0.98, 0.99, 0.995, 0.999]
SMOKE = False   # reads the predictions written by notebook 03 in smoke mode
PROJECT_DIR = ".."

## 0. Setup

In [2]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path(PROJECT_DIR).resolve()
sys.path.insert(0, str(ROOT / "src"))
from redactner.baselines import load_predictions  # noqa: E402
from redactner.inference import TokenClassifierDetector  # noqa: E402
from redactner.labels import LABELS  # noqa: E402
from redactner.metrics import CorpusScorer, TabScorer, tab_ignored_chars  # noqa: E402

SUFFIX = "_smoke" if SMOKE else ""
DATA = ROOT / "data" / "processed"
PREDICTIONS = ROOT / "outputs" / f"predictions{SUFFIX}" / "selected"
RESULTS = ROOT / f"results{SUFFIX}"
TEST_SETS = ["test_nemotron", "test_ai4privacy", "test_gretel", "test_tab"]
SET_NAMES = {"test_nemotron": "Nemotron-PII", "test_ai4privacy": "ai4privacy OpenPII", "test_gretel": "Gretel Finance",
             "test_tab": "TAB"}
CATEGORY_NAMES = {"PERSON": "Person", "EMAIL": "Email", "PHONE": "Phone", "ADDRESS": "Address",
                  "LOCATION": "Location", "DATE": "Date", "ID_NUMBER": "ID Number", "FINANCIAL": "Financial",
                  "USERNAME": "Username", "CREDENTIAL": "Credential", "IP_ADDRESS": "IP Address", "URL": "URL",
                  "ORGANIZATION": "Organisation"}


def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


# test documents and the predictions of the exported model saved by notebook 03
sets = {name: read_jsonl(DATA / f"{name}.jsonl") for name in TEST_SETS}
preds = {name: load_predictions(PREDICTIONS / f"{name}.jsonl", docs, "model") for name, docs in sets.items()}
missing = [SET_NAMES[n] for n, p in preds.items() if p is None]
assert not missing, f"Run notebook 03 first; predictions missing for: {', '.join(missing)}"
meta = json.loads((ROOT / "models" / f"redactner-best{SUFFIX}" / "redactner_meta.json").read_text(encoding="utf-8"))
print(f"Model: {meta['run_name']} | predictions for {sum(len(d) for d in sets.values()):,} test documents")

Model: deberta-v3-base_s42 | predictions for 3,127 test documents


## 1. Gretel Label Audit
Fifty Gretel Finance test documents were drawn at random (seed 2026). Each gold span was checked against
Gretel's own annotation scheme, and so was every prediction of the prototype model that overlapped no gold span
(candidates for personal information the annotation missed). Personal information that neither marked was noted
while reading. The verdicts are stored in `results/gretel_audit.csv`.

| Verdict | Meaning |
|---|---|
| OK | A valid value of its category |
| Boundary | The right value with a wrong extent, such as `-03-` for a whole date |
| Label | The right value in the wrong category, such as an account number labelled as a person |
| Not PII | Not a valid instance of its category, such as *monthly*, *30 years* or a field label |
| Missed | Personal information that the gold annotation lacks |

In [3]:
# verdict counts from the manual audit of 50 Gretel Finance test documents
audit = pd.read_csv(ROOT / "results" / "gretel_audit.csv", keep_default_na=False)
VERDICTS = {"OK": "OK", "BOUNDARY": "Boundary", "LABEL": "Label", "NOT_PII": "Not PII", "MISSED": "Missed"}
ORIGINS = {"gold": "Gold Spans", "model": "Model-Only Predictions", "neither": "Marked by Neither"}
counts = audit.groupby(["origin", "verdict"]).size().unstack(fill_value=0).reindex(columns=list(VERDICTS), fill_value=0)
counts = counts.rename(index=ORIGINS, columns=VERDICTS).reindex([ORIGINS[o] for o in ORIGINS])
counts["Total"] = counts.sum(axis=1)
counts.index.name = "Reviewed Items"
counts

verdict,OK,Boundary,Label,Not PII,Missed,Total
Reviewed Items,,,,,,
Gold Spans,236,9,8,51,0,304
Model-Only Predictions,0,5,0,34,57,96
Marked by Neither,0,0,0,0,11,11


In [4]:
# how reliable Gretel's automatic labels are, estimated from the audit
gold = audit[audit.origin == "gold"]
valid_gold = int(gold.verdict.isin(["OK", "BOUNDARY", "LABEL"]).sum())
missed = int((audit.verdict == "MISSED").sum())
model_only = audit[audit.origin == "model"]
summary_rows = {
    "Gold spans that are not personal information (%)": 100 * (gold.verdict == "NOT_PII").mean(),
    "Gold spans with a wrong extent or category (%)": 100 * gold.verdict.isin(["BOUNDARY", "LABEL"]).mean(),
    "Estimated share of the documents' PII that the gold marks (%)": 100 * valid_gold / (valid_gold + missed),
    "Model-only predictions that are real PII (%)": 100 * (model_only.verdict == "MISSED").mean(),
}
pd.Series(summary_rows, name="Value").round(1).to_frame()

,Value
Gold spans that are not personal information (%),16.8
Gold spans with a wrong extent or category (%),5.6
Estimated share of the documents' PII that the gold marks (%),78.8
Model-only predictions that are real PII (%),59.4


Most invalid gold spans are periods and frequencies annotated as dates. The most frequent ones:

In [5]:
# the most frequent gold spans judged not to be personal information
not_pii = gold[gold.verdict == "NOT_PII"]
pd.DataFrame(Counter(zip(not_pii.text.str.lower(), not_pii.label)).most_common(12),
             columns=["Text and Category", "Occurrences"]).assign(
    **{"Text and Category": lambda d: [f"{t} ({CATEGORY_NAMES[l]})" for t, l in d["Text and Category"]]})

,Text and Category,Occurrences
0,monthly (Date),7
1,the year (Date),4
2,semi-annually (Date),2
3,today (Date),2
4,year (Date),2
5,1-year (Date),2
6,franchise business (Organisation),2
7,-03-31 (Date),1
8,tax year (Date),1
9,over 50 years (Date),1


### Scores Against the Corrected Annotation
The exported model is scored on the 50 audited documents twice: against the original Gretel annotation and
against a corrected one, in which invalid gold spans are removed and the missed personal information is added.
Values that neither the annotation nor the prototype marked (mostly placeholder account numbers in SWIFT
messages) are added at every occurrence.

In [6]:
by_id = {d["id"]: d for d in sets["test_gretel"]}
pred_by_id = {d["id"]: p for d, p in zip(sets["test_gretel"], preds["test_gretel"])}
audited_ids = list(dict.fromkeys(audit.doc_id))


def corrected_spans(doc_id):
    """Gold spans with invalid ones removed and the personal information found in the audit added."""
    doc, rows = by_id[doc_id], audit[audit.doc_id == doc_id]
    invalid = {(int(r.start), int(r.end)) for r in rows.itertuples() if r.origin == "gold" and r.verdict == "NOT_PII"}
    spans = [s for s in doc["spans"] if (s[0], s[1]) not in invalid]
    spans += [[int(r.start), int(r.end), r.label] for r in rows.itertuples() if r.origin == "model" and r.verdict == "MISSED"]
    for r in rows.itertuples():
        if r.origin == "neither":
            spans += [[m.start(), m.end(), r.label] for m in re.finditer(re.escape(r.text), doc["text"])]
    return spans


# score the exported model against the original and the corrected labels
rescored = {}
for version, gold_fn in (("Original Annotation", lambda i: by_id[i]["spans"]), ("Corrected Annotation", corrected_spans)):
    sc = CorpusScorer(LABELS)
    for doc_id in audited_ids:
        sc.add(by_id[doc_id]["text"], gold_fn(doc_id), [(s.start, s.end, s.label) for s in pred_by_id[doc_id]])
    s = sc.summary()
    rescored[version] = {"Char Precision": s["char_precision"], "Char Recall": s["char_recall"],
                         "Leak Rate (%)": 100 * s["leak_rate"], "Direct Leak Rate (%)": 100 * s["direct_leak_rate"],
                         "Gold Mentions": s["mentions"]}
audit_scores = pd.DataFrame(rescored).T.round(3)
audit_scores

,Char Precision,Char Recall,Leak Rate (%),Direct Leak Rate (%),Gold Mentions
Original Annotation,0.702,0.990,1.645,1.351,304.0
Corrected Annotation,0.680,0.879,16.619,17.910,349.0


## 2. What Still Leaks
Leaked mentions of the exported model per category, as counts (share of that category's mentions).

In [7]:
# a mention leaks if any letter or digit stays visible; on TAB each unique annotated mention counts once
leak_table, leaked = {}, {}
for name in TEST_SETS:
    rows = []
    for doc, spans in zip(sets[name], preds[name]):
        masked = {i for s in spans for i in range(s.start, s.end)}
        golds = ([m[:3] for ann in doc["annotations"].values() for m in ann] if "annotations" in doc else doc["spans"])
        ignore = tab_ignored_chars(doc["text"]) if "annotations" in doc else set()  # as in the TAB scores
        for s, e, lab in {tuple(g) for g in golds}:
            hidden = all(i in masked for i in range(s, e) if doc["text"][i].isalnum() and i not in ignore)
            rows.append((lab, hidden, doc["id"], s, e))
    df = pd.DataFrame(rows, columns=["label", "hidden", "doc_id", "start", "end"])
    leaked[name] = df[~df.hidden]
    per_label = df.groupby("label").hidden.agg(total="size", hidden="sum")
    leak_table[SET_NAMES[name]] = [f"{int(t - h)} ({100 * (t - h) / t:.1f} %)" if t else ""
                                   for t, h in per_label.reindex(LABELS).fillna(0).itertuples(index=False)]
leak_df = pd.DataFrame(leak_table, index=[CATEGORY_NAMES[l] for l in LABELS]).replace("0 (0.0 %)", "0")
leak_df.index.name = "Category"
leak_df

,Nemotron-PII,ai4privacy OpenPII,Gretel Finance,TAB
Category,,,,
Person,1 (0.1 %),1 (0.1 %),27 (1.7 %),55 (4.9 %)
Email,1 (0.2 %),0,2 (0.6 %),
Phone,0,0,0,
Address,0,2 (0.3 %),15 (1.9 %),
Location,9 (1.8 %),2 (0.4 %),0,199 (37.3 %)
Date,12 (1.1 %),0,33 (1.7 %),152 (5.5 %)
ID Number,0,0,0,226 (63.1 %)
Financial,0,0,1 (0.6 %),
Username,0,,0,


On TAB, every unique mention marked by any annotator is counted once here. TAB's annotators also masked public
bodies and countries when these could help identify the applicant. The taxonomy used here maps countries to
non-personal information in every training set and does not treat public institutions as personal, so part of
TAB's organisation and location leaks reflects this difference in policy rather than a detection error.

Examples of leaked TAB mentions with their context:

In [8]:
# random leaked TAB mentions with 60 characters of context on each side
tab_by_id = {d["id"]: d for d in sets["test_tab"]}
examples = leaked["test_tab"].sample(min(N_EXAMPLES, len(leaked["test_tab"])), random_state=SEED)
pd.DataFrame([{"Category": CATEGORY_NAMES[r.label], "Leaked Text": tab_by_id[r.doc_id]["text"][r.start:r.end],
               "Context": "…" + tab_by_id[r.doc_id]["text"][max(0, r.start - 60):r.end + 60].replace("\n", " ") + "…"}
              for r in examples.itertuples()])

,Category,Leaked Text,Context
0,Location,United Kingdom,…disenfranchisement as a result of his residen...
1,ID Number,29280/95,… the same complaints (applications nos. 26480...
2,Location,the Galleries,…orthumbria Police Authority. There was a post...
3,ID Number,1,"…lots nos. 60/1/2, 63/1 and 63/3, sheet/plan 1..."
4,ID Number,22/2,"…with bore hole, area: 4,877 sq. m; (f) plots ..."
5,Organisation,House of Lords,… applicants were informed by letter of 18 May...
6,Date,ten years,"…a threat to kill, the applicant was sentenced..."
7,Date,several weeks later,…g first dragged her past her father’s body. H...
8,Organisation,Gaziantep Administrative Court,…ty or vital interests of the State. On 7 Dec...
9,ID Number,65723/01,…PROCEDURE The case originated in an applicat...


Leaked TAB mentions by shape (digits replaced by `0`, letters by `a` or `A`), which groups systematic misses:

In [9]:
def shape(text):
    """Reduce text to its shape (digits to 0, letters to a or A) to group similar leaks."""
    return re.sub(r"[A-Z]+", "A", re.sub(r"[a-z]+", "a", re.sub(r"\d", "0", text)))


tab_shapes = Counter((CATEGORY_NAMES[r.label], shape(tab_by_id[r.doc_id]["text"][r.start:r.end]))
                     for r in leaked["test_tab"].itertuples())
# court application numbers such as 36110/97
app_numbers = [(d, s, e) for d in sets["test_tab"] for ann in d["annotations"].values() for s, e, lab, *_ in ann
               if lab == "ID_NUMBER" and re.fullmatch(r"\d{3,6}/\d{2}", d["text"][s:e])]
leaked_app = sum(1 for r in leaked["test_tab"].itertuples() if r.label == "ID_NUMBER"
                 and re.fullmatch(r"\d{3,6}/\d{2}", tab_by_id[r.doc_id]["text"][r.start:r.end]))
print(f"Court application numbers (such as 36110/97): {leaked_app} of "
      f"{len({(d['id'], s, e) for d, s, e in app_numbers})} unique mentions leaked")
pd.DataFrame([{"Category": c, "Shape": s, "Leaked Mentions": n} for (c, s), n in tab_shapes.most_common(12)])

Court application numbers (such as 36110/97): 166 of 286 unique mentions leaked


,Category,Shape,Leaked Mentions
0,ID Number,00000/00,161
1,Location,Aa,113
2,Organisation,Aa Aa Aa,69
3,Organisation,Aa Aa,65
4,Organisation,Aa,52
5,Organisation,A,41
6,Date,a a,35
7,Organisation,Aa a Aa,29
8,Location,Aa Aa,27
9,Person,Aa Aa Aa,16


## 3. Disagreements on TAB
TAB's annotators masked only what identifies the applicant, so judges, government agents and public bodies were
deliberately left visible. The model redacts every name, date and organisation, which costs precision on TAB but
follows the policy the application is built for. The most frequent redacted text that no annotator masked:

In [10]:
# text the model redacted that no TAB annotator masked
extra = Counter()
for doc, spans in zip(sets["test_tab"], preds["test_tab"]):
    covered = {i for ann in doc["annotations"].values() for s, e, *_ in ann for i in range(s, e)}
    for sp in spans:
        if not any(i in covered for i in range(sp.start, sp.end)):
            extra[(doc["text"][sp.start:sp.end], CATEGORY_NAMES.get(sp.label, sp.label))] += 1
pd.DataFrame([{"Redacted Text": t, "Category": c, "Occurrences": n} for (t, c), n in extra.most_common(15)])

,Redacted Text,Category,Occurrences
0,European Commission of Human Rights,Organisation,43
1,born,Date,41
2,Commission,Organisation,37
3,State Security Court,Organisation,32
4,first applicant,Person,27
5,Foreign and Commonwealth Office,Organisation,27
6,United Kingdom of Great Britain and Northern I...,Organisation,27
7,second applicant,Person,27
8,applicant,Person,25
9,United Kingdom Government,Organisation,25


## 4. The Operating Point on Real Documents
The decision threshold was tuned on synthetic development data, on which the model is confident. On real court
judgments it is less certain, so a stricter threshold recovers many of the missed identifiers at the cost of
precision. The table varies the threshold of the exported model on TAB and on the development set. It is a
sensitivity analysis only: the exported threshold was fixed on the development set before any test set was scored.

In [11]:
# re-decode TAB and the development set at stricter thresholds (a sensitivity analysis only)
det = TokenClassifierDetector(ROOT / "models" / f"redactner-best{SUFFIX}")
dev_docs = read_jsonl(DATA / "dev.jsonl")[: 20 if SMOKE else None]
tab_probs = [det.token_probs(d["text"]) for d in sets["test_tab"]]
dev_probs = [det.token_probs(d["text"]) for d in dev_docs]
tab_ignore = [tab_ignored_chars(d["text"]) for d in sets["test_tab"]]
sens_rows = []
for tau in TAU_SENSITIVITY:
    sc, tab, dev_sc = CorpusScorer(LABELS), TabScorer(), CorpusScorer(LABELS)
    for d, tp, ignore in zip(sets["test_tab"], tab_probs, tab_ignore):
        pred = [(s.start, s.end, s.label) for s in det.decode(d["text"], tp, tau)]
        for mentions in d["annotations"].values():
            sc.add(d["text"], [m[:3] for m in mentions], pred, ignore=ignore)
        tab.add(d["text"], d["annotations"], pred, ignore=ignore)
    for d, tp in zip(dev_docs, dev_probs):
        dev_sc.add(d["text"], d["spans"], [(s.start, s.end, s.label) for s in det.decode(d["text"], tp, tau)])
    s, t, dv, per = sc.summary(), tab.summary(), dev_sc.summary(), sc.per_label()
    sens_rows.append({"τ": f"{tau}{' (exported)' if tau == det.threshold else ''}",
                      "TAB Direct Leak Rate (%)": 100 * s["direct_leak_rate"],
                      "TAB Person Leak Rate (%)": 100 * per["PERSON"]["leak_rate"],
                      "TAB ID Number Leak Rate (%)": 100 * per["ID_NUMBER"]["leak_rate"],
                      "TAB Entity Recall, Direct": t["er_direct"], "TAB Word Precision": t["word_precision"],
                      "Development Char Precision": dv["char_precision"],
                      "Development Leak Rate (%)": 100 * dv["leak_rate"]})
del det
sensitivity = pd.DataFrame(sens_rows).set_index("τ").round(3)
sensitivity

,TAB Direct Leak Rate (%),TAB Person Leak Rate (%),TAB ID Number Leak Rate (%),"TAB Entity Recall, Direct",TAB Word Precision,Development Char Precision,Development Leak Rate (%)
τ,,,,,,,
0.97 (exported),21.548,3.142,68.797,0.872,0.678,0.894,0.579
0.98,18.661,2.513,60.112,0.929,0.655,0.876,0.478
0.99,15.704,1.257,52.792,0.970,0.610,0.838,0.304
0.995,13.635,0.749,46.712,0.988,0.562,0.796,0.251
0.999,0.139,0.097,0.248,1.000,0.453,0.675,0.131


## 5. Prototype Versus Final Model on TAB
The prototype checkpoint was trained on a smaller set without Gretel Finance or augmentation. Leak rates per
category on TAB, each model at its own tuned threshold, show which categories changed between the two training sets.

In [12]:
# per-category TAB leak rates of the prototype and the exported model, from notebook 03's results
metrics = json.loads((RESULTS / "metrics.json").read_text(encoding="utf-8"))
suffix = ", Per-Category τ" if metrics["use_category_tau"] else ", Global τ"
final_key = next(k for k in metrics["per_label"] if k.startswith("test_tab|Selected:") and k.endswith(suffix))
proto_key = next((k for k in metrics["per_label"] if k.startswith("test_tab|") and "Prototype" in k), None)
if proto_key:
    compare = pd.DataFrame({
        "Mentions": {CATEGORY_NAMES[l]: v["support"] for l, v in metrics["per_label"][final_key].items()},
        "Prototype Leak Rate (%)": {CATEGORY_NAMES[l]: 100 * v["leak_rate"]
                                    for l, v in metrics["per_label"][proto_key].items() if v["leak_rate"] is not None},
        "Final Model Leak Rate (%)": {CATEGORY_NAMES[l]: 100 * v["leak_rate"]
                                     for l, v in metrics["per_label"][final_key].items() if v["leak_rate"] is not None},
    }).dropna().round(2)
    compare.index.name = "Category"
    display(compare)  # noqa: F821
else:
    print("The prototype checkpoint was not evaluated in notebook 03.")

,Mentions,Prototype Leak Rate (%),Final Model Leak Rate (%)
Category,,,
Person,4138,4.88,3.07
Location,1459,20.01,30.50
Date,9536,6.03,3.16
ID Number,1612,8.50,68.55
Organisation,1946,50.98,33.45


In [13]:
# save the error analysis for the report
with open(RESULTS / "error_analysis.json", "w", encoding="utf-8") as fh:
    json.dump({"gretel_audit": {k: round(v, 2) for k, v in summary_rows.items()},
               "gretel_audit_counts": counts.to_dict(), "gretel_rescored": audit_scores.to_dict(orient="index"),
               "leaks_by_category": leak_df.to_dict(), "tab_threshold_sensitivity": sensitivity.reset_index().to_dict(orient="records")},
              fh, indent=2, default=str)
print("Saved", (RESULTS / "error_analysis.json").relative_to(ROOT).as_posix())

Saved results/error_analysis.json
